# Retrieval Augmented Generation

RAG & RAG + Graph systems for evaluation of LLMs

In [ ]:
!nvidia-smi -L

To get around annoying text-wrapping issues, run this cell.

In [ ]:
from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)

# Mounting drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd "your/data/path"


# Installing Dependencies

In [ ]:
!apt-get update
!apt-get install -y build-essential cmake git libcuda1-525 libcudnn8-dev ninja-build

In [ ]:
!pip install --upgrade pip

!pip install -q sentence-transformers faiss-gpu-cu12 networkx torch transformers
!pip install -q rouge-score nltk bert-score evaluate scikit-learn scikit-build-core numpy
!pip install -q accelerate bitsandbytes
!pip install -q cmake ninja json-repair pydantic
!pip install -q datapizza-ai datapizza-ai-parsers-docling datapizza-ai-embedders-fastembedder

# RAG System

In [ ]:
import json
import torch
import numpy as np
import csv
import os
import uuid
from typing import List, Dict, Any, Tuple, Optional, Union, Callable
from abc import ABC, abstractmethod
from fastembed import TextEmbedding
import faiss
import logging
from transformers import AutoTokenizer, AutoModel, pipeline
import warnings
warnings.filterwarnings("ignore")

import networkx as nx
HAS_NETWORKX = True

from sentence_transformers import SentenceTransformer
import faiss
HAS_FAISS = True

from datapizza.core.vectorstore import VectorConfig
from datapizza.embedders.fastembedder import FastEmbedder
from datapizza.vectorstores.qdrant import QdrantVectorstore
from datapizza.type import Chunk, DenseEmbedding, SparseEmbedding
from qdrant_client import QdrantClient
HAS_DATAPIZZA = True

In [ ]:
# @title Select Embedding Model
selected_emb = 'bge' # @param ["bge", "fast", "accurate", "best", "bge-large"]

embedding_models_dic = {
    "fast": "sentence-transformers/all-mpnet-base-v2",
    "accurate": "intfloat/e5-base-v2",
    "best": "intfloat/multilingual-e5-large-instruct",
    "bge-large": "BAAI/bge-base-en-v1.5",
    "bge": "BAAI/llm-embedder"
}

EMBEDDING_MODEL = embedding_models_dic[selected_emb]

In [ ]:
# @title Configuration

GRAPH_FILENAMES = {"GDPR": "gdpr_w_annexes.json", "AI ACT": "ai-act.json" }

class Config:
  def __init__(self, domain: str, selected_emb: str, graph_filename: list, max_tokens: int = 256, use_cache: bool = False, topk: int = 20):
    self.DOMAIN = domain
    self.GRAPH_FILENAME = graph_filename
    self.GRAPH_FILEPATH = os.path.join(domain, "datasets", graph_filename)

    self.DATASET_FILENAME = "dataset_truncated.jsonl" if domain == "GDPR" else "dataset_truncated.json"
    self.DATASET_JSONL = os.path.join(domain, "datasets", self.DATASET_FILENAME)
    self.SHUFFLED_DATASET = os.path.join(domain, "datasets", "shuffled_" + self.DATASET_FILENAME)

    self.EMBEDDING_MODEL = embedding_models_dic[selected_emb]
    self.EMBEDDINGS_DIR = os.path.join(domain, "embeddings", selected_emb)
    self.RETRIEVAL_RESULTS_DIR = os.path.join(domain, "retrieval_results", selected_emb)

    self.TOPK = topk
    self.USE_CACHE = use_cache
    self.MAX_TOKENS = max_tokens

    self.GENERATOR_RESULTS_DIR = os.path.join(domain, "generator_results")

gdpr_config = Config("GDPR", selected_emb, GRAPH_FILENAMES["GDPR"], max_tokens=256, use_cache=False)
aiact_config = Config("AI ACT", selected_emb, GRAPH_FILENAMES["AI ACT"], max_tokens=256, use_cache=False)

In [ ]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class AbstractRAG(ABC):
    """Abstract base class for all RAG implementations."""

    @abstractmethod
    def load_documents(self, json_file: str, max_tokens: int = 128) -> None:
        pass

    @abstractmethod
    def embed_documents(self, savepath: Optional[str] = None) -> None:
        pass

    @abstractmethod
    def load_embeddings(self, *args, **kwargs) -> None:
        pass

    @abstractmethod
    def retrieve(self, query: str, topk: int = 20, threshold: float = 0.0,
                query_embedding: Optional[np.ndarray] = None) -> List[Dict[str, Any]]:
        pass

    @abstractmethod
    def build_graph(self) -> None:
        pass

    @abstractmethod
    def retrieve_with_graph(self, query: str, topk: int = 5, threshold: float = 0.6,
                           query_embedding: Optional[np.ndarray] = None) -> List[Dict[str, Any]]:
        pass

In [ ]:
def load_dataset(jsonl_file: str) -> List[Dict]:
    qa_pairs = []
    with open(jsonl_file, 'r') as f:
        for line in f:
            if not line.strip(): continue
            data = json.loads(line)
            if 'question' in data and 'answer' in data:
                qa_pairs.append({
                    'question': data['question'],
                    'answer': data['answer'],
                    'refid': data.get('ref_id', ''),
                    'type': data.get('type', ''),
                    'recital_number': data.get('recital_number', ''),
                    'article_number': data.get('article_number', ''),
                    'annex_number': data.get('annex_number', '')
                })
    return qa_pairs

def sort_index_related_lists(x:List, y:List):
   # Zip them together, sort, and unzip
    combined = sorted(zip(x, y), key=lambda x: x[0], reverse=True)
    x_sorted, y_sorted = zip(*combined)

    # Result is tuples, convert back to lists if needed
    return list(x_sorted), list(y_sorted)

In [ ]:
def get_timestamp():
  from datetime import datetime
  return datetime.now().strftime("%Y%m%d_%H%M%S")

In [ ]:
def chunk_text(text: str, max_tokens: int = 128, overlap: int = 50) -> List[str]:
    """Segment text into chunks with overlap."""
    try:
        from transformers import AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained("BAAI/llm-embedder")
        tokens = tokenizer.encode(text, add_special_tokens=False)
        chunks = []
        for i in range(0, len(tokens), max_tokens - overlap):
            chunk_tokens = tokens[i:i + max_tokens]
            chunks.append(tokenizer.decode(chunk_tokens))
        return chunks
    except:
        # Fallback: simple character chunking
        chunks = []
        for i in range(0, len(text), max_tokens * 4):  # Rough token approximation
            chunks.append(text[i:i + max_tokens * 4])
        return [c.strip() for c in chunks if c.strip()]

def load_documents_json(json_file: str, max_tokens: int = 128) -> List[Dict[str, Any]]:
    """
    Minimal robust loader compatible with existing RAGSystem schema.
    - Keeps your AI-ACT -> articles flattening logic.
    - Ensures extra_fn(item) is ALWAYS a dict (never a list), so **extra_fn(item) is safe.
    """
    with open(json_file, "r") as f:
        data = json.load(f)

    # ---- 1) Flatten AI-ACT chapters/sections into data["articles"] (same as your code) ----
    for chapter in data.get("chapters", []):  # Align ai-act format to gdpr [file:1]
        if not data.get("articles", []):
            data["articles"] = []

        for article in chapter.get("articles", []):
            data["articles"].append({
                "number": article["number"],
                "fullText": article["fullText"],
                "relatedRecitals": article.get("relatedRecitals", []),
            })

        for section in chapter.get("sections", []):
            for article in section.get("articles", []):
                data["articles"].append({
                    "number": article["number"],
                    "fullText": article["fullText"],
                    "relatedRecitals": article.get("relatedRecitals", []),
                })

    # ---- 2) Same doc_configs but with safe extra handling ----
    doc_configs = [
        ("articles", "article",
         lambda x: x.get("fullText", ""),
         lambda x: {"relatedRecitals": x.get("relatedRecitals", [])}),
        ("annexes", "annex",
         lambda x: x.get("fullText", x.get("text", "")),
         lambda x: {"relatedArticles": x.get("articles", []),
                    "relatedRecitals": x.get("recitals", [])}),
        ("recitals", "recital",
         lambda x: x.get("text", ""),
         lambda x: {}),
    ]  # matches your intent [file:1]

    documents: List[Dict[str, Any]] = []

    def safe_extra(extra: Any) -> Dict[str, Any]:
        # Critical fix: **extra must be mapping [file:1]
        if extra is None:
            return {}
        if isinstance(extra, dict):
            return extra
        # If it is a list/tuple/etc, don't explode: store it under a key or drop it
        return {"_extra": extra}

    for key, doc_type, text_fn, extra_fn in doc_configs:
        if key not in data:
            continue

        for item in data[key]:
            # Hard guard: item MUST be dict because you call item.get(...) and item['number'] [file:1]
            if not isinstance(item, dict):
                # If your dataset ever contains non-dict entries, skip instead of crashing
                continue

            text = text_fn(item)
            if not text:
                continue

            num = item.get("number", None)
            if num is None:
                continue

            parent = f"{doc_type.capitalize()}{num}"
            title = item.get("title", f"{doc_type.capitalize()} {num}")

            chunks = chunk_text(text, max_tokens=max_tokens)  # your existing chunker [file:1]
            extra = safe_extra(extra_fn(item))

            for i, chunk in enumerate(chunks):
                final_text = title + " " + chunk
                doc_id = f"{doc_type.capitalize()}_{num}_chunk{i}"
                documents.append({
                    "id": doc_id,
                    "parent_id": parent,
                    "type": doc_type,
                    "number": num,
                    "text": final_text,
                    "title": title,
                    **extra,
                })

    logger.info(f"Loaded {len(documents)} chunks from {json_file}")
    return documents

In [ ]:
def parse_refid_to_ground_truth(ref_id: str, question_type: str):
    gt = {"article": False, "annex": False, "recital": False}

    # prende solo i pezzi numerici del ref_id (es: "1_4_aug_0" -> ["1","4","0"])
    nums = [t for t in (ref_id or "").split("_") if t.isdigit()]

    if "unity" in (question_type or ""):
        # unity ha un solo doc target: prende il primo numero
        if "article" in question_type and nums: gt["article"] = nums[0]
        if "annex" in question_type and nums:   gt["annex"] = nums[0]
        if "recital" in question_type and nums: gt["recital"] = nums[0]

    elif "binding" in (question_type or ""):
        # binding: assumo che i primi due numeri siano i due riferimenti (es: annex=1, article=4)
        if len(nums) >= 2:
            a, b = nums[0], nums[1]
            if "article_recital" in question_type:
                gt["article"], gt["recital"] = a, b
            elif "annex_article" in question_type:
                gt["annex"], gt["article"] = a, b
            elif "annex_recital" in question_type:
                gt["annex"], gt["recital"] = a, b

    return gt


def extract_id_from_doc(doc_id: str) -> tuple[str, str]:
    """
    Extract parent document type and number from chunk ID.
    'Article_49_chunk2' -> ('article', '49')
    'Annex_7_chunk0' -> ('annex', '7')
    """
    if doc_id.startswith('Article_'):
        parts = doc_id.split('_')
        return ('article', parts[1])
    elif doc_id.startswith('Annex_'):
        parts = doc_id.split('_')
        return ('annex', parts[1])
    elif doc_id.startswith('Recital_'):
        parts = doc_id.split('_')
        return ('recital', parts[1])
    return (None, None)

def create_retrieval_cache(rag_system, qa_pairs, top_k=20, use_graph=False, output_file="cache.jsonl", threshold=0.6):
    """Generate rag cache"""
    cache = []
    for pair in qa_pairs:
        question, refid, question_type = pair['question'], pair.get('refid', ''), pair.get('type', '')

        if not refid:
          recital_number = pair.get("recital_number", False)
          article_number = pair.get("article_number", False)
          annex_number = pair.get("annex_number", False)
          if recital_number or article_number or annex_number:
            gt = {"article": article_number, "annex": annex_number, "recital": recital_number}
          else:
            continue
        else:
          gt = parse_refid_to_ground_truth(refid, question_type)

        context = rag_system.retrieve_with_graph(question, top_k, threshold=threshold) if use_graph else rag_system.retrieve(question, top_k, threshold=threshold)

        predictions, similarities = [], []
        for doc in context:
            doc_type, doc_num = extract_id_from_doc(doc['id'])
            graph_related = doc.get('graph_related', False)
            if doc_type:
                predictions.append({doc_type: doc_num, "graph_related": graph_related})
                similarities.append(doc['score'])

        # Sort similarities
        if not similarities or not predictions:
            print(f"Empty predictions or similarity")
            continue
        similarities, predictions = sort_index_related_lists(similarities, predictions)

        cache.append({"question": question, "ground_truth": gt, "predictions": predictions, "similarity": similarities, "threshold": threshold})

    with open(output_file, 'w') as f:
        for entry in cache: f.write(json.dumps(entry) + '\n')
    return cache

def calculate_retrieval(rag_system, qa_pairs: List[Dict], top_k: int = 20,
                      use_graph: bool = False, cache_file: str = None, load_cache: bool = False, threshold: float = float("-inf")):
    """Pure retrieval."""
    if load_cache and cache_file and os.path.exists(cache_file):
        logger.info(f"Loading existing cache: {cache_file}")
        cache = []
        with open(cache_file) as f:
            for line in f: cache.append(json.loads(line))
    else:
        cache = create_retrieval_cache(rag_system, qa_pairs, top_k, use_graph, cache_file, threshold)

    return cache  # Returns retrieval cache for metrics

In [ ]:
class BasicRAG(AbstractRAG):
    """Basic FAISS + SentenceTransformer implementation."""

    def __init__(self, embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2", domain: str = "generic"):
        self.embedding_model = embedding_model
        self.model = None
        self.documents: List[Dict[str, Any]] = []
        self.embeddings: Optional[np.ndarray] = None
        self.index = None
        self.graph = None if HAS_NETWORKX else None
        self.domain = domain
        self._load_model()

    def _load_model(self):
        """Load embedding model."""
        try:
            # Try BAAI/llm-embedder first
            self.model = SentenceTransformer(self.embedding_model)
            logger.info(f"Loaded TextEmbedding model: {self.embedding_model}")
        except:
            self.model = TextEmbedding(model_name=self.embedding_model)
            logger.info(f"Loaded SentenceTransformer: {self.embedding_model}")

    def load_documents(self, json_file: str, max_tokens: int = 128) -> None:
        self.documents = load_documents_json(json_file, max_tokens)

    def embed_documents(self, savepath: Optional[str] = None) -> None:
        if not self.documents:
            raise ValueError("Load documents first!")

        texts = [doc['text'] for doc in self.documents]
        self.embeddings = self.model.encode(
            texts, batch_size=16, normalize_embeddings=True, show_progress_bar=True
        ).astype(np.float32)

        dimension = self.embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dimension)
        faiss.normalize_L2(self.embeddings)
        self.index.add(self.embeddings)

        logger.info(f"Created embeddings with shape {self.embeddings.shape}")
        if savepath:
            self._save_embeddings_csv(savepath)

    def load_embeddings(self, articles_csv: Optional[str] = None, annexes_csv: Optional[str] = None,
                       recitals_csv: Optional[str] = None) -> None:
        """Load pre-computed embeddings from CSV files."""
        all_docs = []
        all_embeddings = []

        csv_files = [
            (articles_csv, 'article'),
            (annexes_csv, 'annex'),
            (recitals_csv, 'recital')
        ]

        for csv_path, doctype in csv_files:
            if csv_path and os.path.exists(csv_path):
                all_docs.extend(self._load_csv_embeddings(csv_path, doctype))

        if not all_docs:
            raise ValueError("No embedding CSV files found or loaded!")

        self.documents = all_docs
        self.embeddings = np.array([doc['embedding'] for doc in all_docs], dtype=np.float32)
        dimension = self.embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dimension)
        faiss.normalize_L2(self.embeddings)
        self.index.add(self.embeddings)

        logger.info(f"Loaded {len(self.documents)} documents from CSV embeddings")

    def _load_csv_embeddings(self, csvpath: str, doctype: str) -> List[Dict[str, Any]]:
        docs = []
        with open(csvpath, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                embedding = np.array(json.loads(row['embedding']), dtype=np.float32)
                docs.append({
                    'id': f"{doctype.capitalize()}{row['number']}",
                    'type': row['type'],
                    'number': row['number'],
                    'text': row['text'],
                    'title': f"{doctype.capitalize()} {row['number']}",
                    'embedding': embedding
                })
        logger.info(f"Loaded {len(docs)} {doctype}s from {csvpath}")
        return docs

    def _save_embeddings_csv(self, outputdir: str) -> None:
        os.makedirs(outputdir, exist_ok=True)
        docs_by_type = {'article': [], 'annex': [], 'recital': []}

        for doc, emb in zip(self.documents, self.embeddings):
            docs_by_type[doc['type']].append(doc)

        embedding_dict = {doc['id']: emb.tolist() for doc, emb in zip(self.documents, self.embeddings)}

        for doctype, docs in docs_by_type.items():
            if not docs:
                continue
            csvpath = os.path.join(outputdir, f"full{doctype}sembeddings.csv")
            with open(csvpath, 'w', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerow(['type', 'number', 'text', 'embedding'])
                for doc in docs:
                    embedding_str = json.dumps(embedding_dict[doc['id']])
                    writer.writerow([doctype, doc['number'], doc['text'], embedding_str])
            logger.info(f"Saved {len(docs)} {doctype}s to {csvpath}")

    def retrieve(self, query: str, topk: int = 20, threshold: float = 0.0,
                query_embedding: Optional[np.ndarray] = None) -> List[Dict[str, Any]]:
        if self.index is None or self.documents is None:
            raise ValueError("Index or documents missing!")

        try:
            if query_embedding is None:
                query_with_prefix = f"Represent this sentence for retrieval of relevant passages: {query}"
                query_embedding = self.model.encode(
                    [query_with_prefix], show_progress_bar=False,
                    normalize_embeddings=True, convert_to_numpy=True
                ).astype(np.float32)

            # ---- CRITICAL: always enforce dtype + 2D shape + L2 norm ----
            query_embedding = np.asarray(query_embedding, dtype=np.float32)
            if query_embedding.ndim == 1:
                query_embedding = query_embedding.reshape(1, -1)
            faiss.normalize_L2(query_embedding)
            # ------------------------------------------------------------

            scores, indices = self.index.search(query_embedding, topk)
            results = []
            for score, idx in zip(scores[0], indices[0]):
                if idx < len(self.documents) and float(score) >= threshold:
                    doc = self.documents[idx]
                    results.append({
                        'id': doc['id'],
                        'text': doc['text'],
                        'score': float(score),
                        'type': doc['type']
                    })

            while len(results) < topk:
                results.append({'id': 'PAD', 'text': '', 'score': -1.0, 'type': 'pad'})
            return results[:topk]
        except Exception as e:
            logger.error(f"RETRIEVE ERROR: {e}")
            return [{'id': 'ERROR', 'text': f'Error: {e}', 'score': -1.0, 'type': 'error'}] * topk

    def build_graph(self) -> None:
        if not HAS_NETWORKX:
            logger.warning("NetworkX not available, skipping graph construction")
            return

        self.graph = nx.DiGraph()
        for doc in self.documents:
            self.graph.add_node(doc['id'], type=doc['type'], text=doc['text'])

        for doc in self.documents:
            source_id = doc['id']
            if doc['type'] == 'article':
                related_recitals = doc.get('relatedRecitals', [])
                for rnum in related_recitals:
                    target_id = f"Recital_{rnum}_chunk0"
                    if self.graph.has_node(target_id):
                        self.graph.add_edge(source_id, target_id, relation='related_recital')
            elif doc['type'] == 'annex':
                related_articles = doc.get('relatedArticles', [])
                for anum in related_articles:
                    target_id = f"Article_{anum}_chunk0"
                    if self.graph.has_node(target_id):
                        self.graph.add_edge(source_id, target_id, relation='related_article')
                related_recitals = doc.get('relatedRecitals', [])
                for rnum in related_recitals:
                    target_id = f"Recital_{rnum}_chunk0"
                    if self.graph.has_node(target_id):
                        self.graph.add_edge(source_id, target_id, relation='related_recital')

        logger.info(f"Built graph: {self.graph.number_of_nodes()} nodes, {self.graph.number_of_edges()} edges")

    def retrieve_with_graph(self, query: str, topk: int = 5, threshold: float = 0.6,
                           query_embedding: Optional[np.ndarray] = None) -> List[Dict[str, Any]]:
        retrieved_docs = self.retrieve(
            query=query,
            topk=topk,
            threshold=threshold,
            query_embedding=query_embedding
        )
        retrieved_docs = [d for d in retrieved_docs if d.get("id") != "PAD"]

        final_results = []
        seen_ids = set()

        def add_doc(doc_obj: Dict[str, Any], is_graph_result: bool = False):
            if doc_obj['id'] not in seen_ids:
                seen_ids.add(doc_obj['id'])
                if is_graph_result:
                    doc_obj['graph_related'] = True
                final_results.append(doc_obj)

        for doc in retrieved_docs:
            add_doc(doc)
            if doc['type'] in ['annex', 'article'] and self.graph and self.graph.has_node(doc['id']):
                neighbors = list(self.graph.neighbors(doc['id']))
                for neighbor_id in neighbors:
                    neighbor_doc = next((d for d in self.documents if d['id'] == neighbor_id), None)
                    if neighbor_doc:
                        neighbor_result = {
                            'id': neighbor_doc['id'],
                            'text': neighbor_doc['text'],
                            'score': doc['score'],
                            'type': neighbor_doc['type']
                        }
                        add_doc(neighbor_result, is_graph_result=True)

        return final_results[:topk] if final_results else [{'id':'PAD','text':'','score':-1.0,'type':'pad'}] * topk

In [ ]:
class DatapizzaRAG(AbstractRAG):
    """Datapizza implementation with Qdrant vector store."""

    def __init__(self, embedding_model: str = "BAAI/bge-base-en-v1.5",
                 vectorstore_path: str = "./qdrant_db",
                 domain:str = "generic" ):
        if not HAS_DATAPIZZA:
            raise ImportError("Datapizza libraries not installed. pip install datapizza-ai[all]")

        self.embedding_model = embedding_model
        self.vectorstore_path = vectorstore_path
        self.documents: List[Dict[str, Any]] = []
        self.model = None
        self.vectorstore = None
        self.graph = None if HAS_NETWORKX else None
        self.domain = domain
        self._setup_datapizza()

    def _setup_datapizza(self):
        """Initialize Datapizza components."""
        self.model = TextEmbedding(
            model_name=self.embedding_model
        )
        self.vectorstore = QdrantVectorstore(location=":memory:")
        # vectorstore = QdrantVectorstore(path=self.vectorstore_path)
        self.vectorstore.create_collection(
            "mydocuments",
            vector_config=[VectorConfig(name="dense_embeddings", dimensions=768)]
        )

    def load_documents(self, json_file: str, max_tokens: int = 128) -> None:
        self.documents = load_documents_json(json_file)

    def embed_documents(self, savepath: Optional[str] = None) -> None:
        if not self.documents:
            raise ValueError("Load documents first!")

        chunks = []
        for doc in self.documents:
            if not doc['text'].strip():
                continue
            embeddings = list(self.model.embed([doc['text']]))
            embedding_array = embeddings[0]  # numpy.ndarray
            dense_embedding = DenseEmbedding(name="dense_embeddings",vector=embedding_array)

            chunk = Chunk(
                id=str(uuid.uuid4()),
                text=doc['text'],
                embeddings=[dense_embedding],
                metadata={
                    'id': doc['id'],
                    'type': doc['type'],
                    'source': self.domain,
                    'number': doc['number'],
                    'title': doc['title'],
                    'related_annexes': doc.get('relatedAnnexes', []),
                    'related_articles': doc.get('relatedArticles', []),
                    'related_recitals': doc.get('relatedRecitals', [])
                }
            )
            chunks.append(chunk)

        self.vectorstore.add(chunks, collection_name="mydocuments")
        logger.info(f"Added {len(chunks)} chunks to Datapizza vectorstore")

    def load_embeddings(self, *args, **kwargs) -> None:
        """Datapizza loads from persistent Qdrant storage automatically."""
        logger.info("Datapizza embeddings loaded from persistent storage")

    def retrieve(self, query: str, topk: int = 20, threshold: float = 0.0,
                query_embedding: Optional[np.ndarray] = None) -> List[Dict[str, Any]]:

        if query_embedding is None:
            query_embeddings = list(self.model.embed([query]))
            query_embedding = query_embeddings[0]

        # Get native Qdrant client from Datapizza
        qdrant_client = self.vectorstore.client
        collection = self.vectorstore.client.get_collection("mydocuments")

        # Native search WITH SCORES
        hits = qdrant_client.query_points(
            collection_name="mydocuments",
            query=query_embedding,
            using="dense_embeddings",
            limit=5,
            with_payload=True,
            with_vectors=False
        )

        results = []
        for hit in hits.points:
            score = hit.score  # ✅ Native score
            if score < threshold:
              continue

            results.append({
                'id': hit.payload.get('id', 'unknown'),
                'text': hit.payload.get('text', ''),
                'score': score,
                'type': hit.payload.get('type', 'unknown')
            })
        return results

    def build_graph(self) -> None:
        # Datapizza graph would use metadata relationships - simplified for now
        if not HAS_NETWORKX:
            logger.warning("NetworkX not available, skipping graph construction")
            return
        logger.info("Datapizza graph construction uses metadata relationships")

    def retrieve_with_graph(self, query: str, topk: int = 5, threshold: float = 0.6,
                           query_embedding: Optional[np.ndarray] = None) -> List[Dict[str, Any]]:
        # For Datapizza, graph retrieval uses metadata expansion
        return self.retrieve(query, topk=topk*2, threshold=threshold, query_embedding=query_embedding)[:topk]


In [ ]:
class RAGFactory:
    """Factory pattern for creating RAG instances."""

    @staticmethod
    def create(
        rag_type: str = "basic",
        embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2",
        use_cache: bool = False,
        embeddings_dir: str = "./embeddings",
        vectorstore_path: str = "./qdrant_db",
        **kwargs
    ) -> AbstractRAG:
        """
        Factory method to create RAG instances.

        Args:
            rag_type: 'basic' or 'datapizza'
            embedding_model: Model name for embeddings
            use_cache: Load cached embeddings if available
            embeddings_dir: Directory for basic RAG cached embeddings
            vectorstore_path: Path for Datapizza Qdrant storage
        """
        if rag_type == "basic":
            if not HAS_FAISS:
                raise ImportError("FAISS not available. pip install faiss-gpu-cu12")
            rag = BasicRAG(embedding_model=embedding_model)
        elif rag_type == "datapizza":
            if not HAS_DATAPIZZA:
                raise ImportError("Datapizza not available. pip install datapizza-ai[all]")
            rag = DatapizzaRAG(embedding_model=embedding_model, vectorstore_path=vectorstore_path)
        else:
            raise ValueError(f"Unknown RAG type: {rag_type}. Choose 'basic' or 'datapizza'")

        return rag

# RAG evaluator

In [ ]:
def init_rag(config: Config, rag_type: str = "basic"):
  """ Rag initialization """

  rag = None

  # Factory instantiation
  if rag_type == "basic":
    rag = RAGFactory.create(rag_type=rag_type, embedding_model=config.EMBEDDING_MODEL, use_cache=config.USE_CACHE, embeddings_dir=config.EMBEDDINGS_DIR, domain=config.DOMAIN)
  elif rag_type == "datapizza":
    rag = RAGFactory.create(rag_type=rag_type, embedding_model=config.EMBEDDING_MODEL, use_cache=config.USE_CACHE, vectorstor_path=config.EMBEDDINGS_DIR, domain=config.DOMAIN)
  else:
    raise Exception(f"rag_type {rag_type} not allowed")

  # Load documents
  rag.load_documents(config.GRAPH_FILEPATH, max_tokens=config.MAX_TOKENS)

  # Embed
  rag.embed_documents(config.EMBEDDINGS_DIR)

  # Build graph
  rag.build_graph()

  # Rag graph
  print("Graph nodes:", rag.graph.number_of_nodes())
  print("Graph edges:", rag.graph.number_of_edges())
  for d in rag.documents[:3]:
    print("Sample node:", d['id'])
    print("Neighbors:", list(rag.graph.neighbors(d['id'])))

  return rag

In [ ]:
qa_pairs_dict = {}

for config in [gdpr_config, aiact_config]:
  if config.USE_CACHE:
    qa_pairs = load_dataset(config.SHUFFLED_DATASET)
  else:
    qa_pairs = np.array(load_dataset(config.DATASET_JSONL))
    np.random.shuffle(qa_pairs)
    qa_pairs = qa_pairs.tolist()

    with open(config.SHUFFLED_DATASET, "w") as f:
      for entry in qa_pairs: f.write(json.dumps(entry) + '\n')
    print(f"Saved shuffled dataset to {config.SHUFFLED_DATASET}")

  qa_pairs_dict[config.DOMAIN] = qa_pairs
  print(f"Loaded {len(qa_pairs)} QA pairs for {config.DOMAIN}")


In [ ]:
# datapizza_gdpr_rag = init_rag(gdpr_config, "datapizza")
# datapizza_aiact_rag = init_rag(aiact_config, "datapizza")

basic_gdpr_rag = init_rag(gdpr_config, "basic")
basic_aiact_rag = init_rag(aiact_config, "basic")

rags = {
    # "gdpr_datapizza": {
    #     "rag": init_rag(gdpr_config, "datapizza"),
    #     "retrieval_path": os.path.join(RETRIEVAL_RESULTS_DIR, "datapizza_rag_cache.jsonl"),
    #     "use_graph": False,
    #     "domain": gdpr_config.DOMAIN
    # },
    # "aiact_datapizza": {
    #     "rag": init_rag(aiact_config, "datapizza"),
    #     "retrieval_path": os.path.join(RETRIEVAL_RESULTS_DIR, "datapizza_rag_cache.jsonl"),
    #     "use_graph": False,
    #     "domain": aiact_config.DOMAIN
    # },
    "gdpr_basic": {
        "rag": basic_gdpr_rag,
        "retrieval_path": os.path.join(gdpr_config.RETRIEVAL_RESULTS_DIR, "basic_rag_cache.jsonl"),
        "use_graph": False,
        "config": gdpr_config
    },
    "gdpr_basic_graph": {
        "rag": basic_gdpr_rag,
        "retrieval_path": os.path.join(gdpr_config.RETRIEVAL_RESULTS_DIR, "basic_ragg_cache.jsonl"),
        "use_graph": True,
        "config": gdpr_config
    },
    "aiact_basic": {
        "rag": basic_aiact_rag,
        "retrieval_path": os.path.join(gdpr_config.RETRIEVAL_RESULTS_DIR, "basic_rag_cache.jsonl"),
        "use_graph": False,
        "config": aiact_config
    },
    "aiact_basic_graph": {
        "rag": basic_aiact_rag,
        "retrieval_path": os.path.join(gdpr_config.RETRIEVAL_RESULTS_DIR, "basic_ragg_cache.jsonl"),
        "use_graph": True,
        "config": aiact_config
    },
}

## Running inference

In [ ]:
for rag_type in rags:
  rag = rags[rag_type]["rag"]

  rag_retrieval_path = rags[rag_type]["retrieval_path"]
  rag_retrieval_folder = os.path.dirname(rag_retrieval_path)

  use_graph = rags[rag_type]["use_graph"]

  config = rags[rag_type]["config"]

  print(f"=== RAG RETRIEVAL {rag_type.upper()} ===")

  if not os.path.exists(rag_retrieval_folder):
    os.makedirs(rag_retrieval_folder, exist_ok=True)

  rag_results = calculate_retrieval(rag, qa_pairs_dict[config.DOMAIN][:], top_k=config.TOPK, use_graph=use_graph, threshold=float("-inf"),
                                    cache_file=rag_retrieval_path, load_cache=config.USE_CACHE)


In [ ]:
import json
import pandas as pd


def load_cache_to_df(
    cache_file: str,
    thresholds=(0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9),
) -> pd.DataFrame:
    """Load cache into DataFrame and expand by thresholds (optimized)."""
    rows = []
    thr_list = list(thresholds)

    with open(cache_file) as f:
        for line in f:
            entry = json.loads(line)
            gt = entry["ground_truth"]
            preds = entry["predictions"]  # list of dicts
            sims = entry["similarity"]    # list of floats

            # Normalize ground truth once
            def norm(v):
                return False if v in ("", None) else v

            ground_article = norm(gt.get("article"))
            ground_annex = norm(gt.get("annex"))
            ground_recital = norm(gt.get("recital"))

            # Pre-zip predictions and sims once
            pred_sim = list(zip(preds, sims))

            # Reuse same zipped list for all thresholds
            for thresh in thr_list:
                filtered_preds = [p for p, s in pred_sim if s >= thresh]

                rows.append(
                    {
                        "threshold": thresh,
                        "ground_article": ground_article,
                        "ground_annex": ground_annex,
                        "ground_recital": ground_recital,
                        "predictions": filtered_preds,
                    }
                )

    return pd.DataFrame(rows)

def compute_recall_table(df: pd.DataFrame, topks=(1, 5, 10, 15, 20)) -> pd.DataFrame:
    """
    Compute recall@k for article/annex/recital with:
    - Separate denominators: only rows that actually have that ground-truth field.
    - Graph-neighbor rule: include graph_related predictions without consuming seed budget k.
    """
    def safe_int(x):
        try:
            if x is False or x is None or x == "":
                return None
            return int(x)
        except (ValueError, TypeError):
            return None

    records = []

    for thr, group in df.groupby("threshold"):
        rows = list(group.itertuples(index=False))

        for k in topks:
            hit_articles = hit_annexes = hit_recitals = 0
            n_articles = n_annexes = n_recitals = 0

            for row in rows:
                ga = row.ground_article
                gx = row.ground_annex
                gr = row.ground_recital

                if ga is not False:
                    n_articles += 1
                if gx is not False:
                    n_annexes += 1
                if gr is not False:
                    n_recitals += 1

                preds = row.predictions or []

                # Build preds_k:
                # - Always include graph_related preds
                # - Include up to k non-graph preds ("seeds")
                preds_k = []
                seed_count = 0
                for pred in preds:
                    is_graph = bool(pred.get("graph_related", False))
                    if is_graph:
                        preds_k.append(pred)
                        continue
                    if seed_count < k:
                        preds_k.append(pred)
                        seed_count += 1
                    else:
                        break

                pred_articles, pred_annexes, pred_recitals = set(), set(), set()

                for p in preds_k:
                    if "article" in p:
                        v = safe_int(p.get("article"))
                        if v is not None:
                            pred_articles.add(v)
                    if "annex" in p:
                        v = safe_int(p.get("annex"))
                        if v is not None:
                            pred_annexes.add(v)
                    if "recital" in p:
                        v = safe_int(p.get("recital"))
                        if v is not None:
                            pred_recitals.add(v)

                if ga is not False:
                    g = safe_int(ga)
                    if g is not None and g in pred_articles:
                        hit_articles += 1

                if gx is not False:
                    g = safe_int(gx)
                    if g is not None and g in pred_annexes:
                        hit_annexes += 1

                if gr is not False:
                    g = safe_int(gr)
                    if g is not None and g in pred_recitals:
                        hit_recitals += 1

            rec_a = (hit_articles / n_articles) if n_articles else 0.0
            rec_x = (hit_annexes / n_annexes) if n_annexes else 0.0
            rec_r = (hit_recitals / n_recitals) if n_recitals else 0.0

            records.append({
                "topk": k,
                "threshold": thr,
                "article": rec_a,
                "annex": rec_x,
                "recital": rec_r,
            })

    return pd.DataFrame(records)


In [ ]:
def print_rag_table(result_df: pd.DataFrame, title:str):
    # Article block
    art = result_df.pivot(index="topk", columns="threshold", values="article").sort_index(ascending=False)
    annex = result_df.pivot(index="topk", columns="threshold", values="annex").sort_index(ascending=False)
    rec = result_df.pivot(index="topk", columns="threshold", values="recital").sort_index(ascending=False)

    print(f"{title} – Article")
    print(art.round(2))
    print(f"\n{title} – Annex")
    print(annex.round(2))
    print(f"\n{title} – Recital")
    print(rec.round(2))

In [ ]:
for rag in rags:
  print(f"=== RAG - {rag.upper()} ===")
  rag_df = load_cache_to_df(rags[rag]["retrieval_path"])
  rag_res = compute_recall_table(rag_df)
  print_rag_table(rag_res, rag.upper())
  print("\n\n")

In [ ]:
res = rags["aiact_basic"]["rag"].retrieve_with_graph("What provisions does the Regulation establish concerning general-purpose AI models? ", topk=20, threshold=float('-inf'))
for d in res[:10]:
    print(d['id'], d['type'], d['score'], getattr(d, 'graph_related', False))

In [ ]:
res = rags["aiact_basic_graph"]["rag"].retrieve("What verification process involving human oversight is required before a deployer can act on the identification results from high-risk AI systems categorized under point 1(a) of Annex III, and in which sectors might this requirement be exempted if it is considered excessive by legislation? ", topk=20, threshold=float('-inf'))
for d in res[:20]:
    print(d['id'], d['type'], d['score'], getattr(d, 'graph_related', False))

# Q&A

## Dependencies

In [ ]:
## INSTALLING HUGGINGFACE
!pip install huggingface-hub

## INSTALLING CMAKE, NINJA, SCIKIT BUILD CORE

## INSTALLING llama-cpp-python
# GPU llama-cpp-python; Starting from version llama-cpp-python==0.1.79, it supports GGUF
!CMAKE_ARGS="-DGGML_CUDA=on  -DCMAKE_CUDA_ARCHITECTURES=80" FORCE_CMAKE=1 pip install llama-cpp-python --force-reinstall --no-cache-dir > /dev/null

## INSTALLING EVALUATORS
!pip install -qq evaluate sacrebleu rouge-score bert-score nltk

In [ ]:
%env CC=/usr/bin/gcc
%env CXX=/usr/bin/g++

In [ ]:
nvidia_model = "A100" # @param ["A100", "T4", "L4"]
dcmake_cuda_ark = {"A100":80, "T4":75, "L4":80}[nvidia_model]

In [ ]:
from typing import List, Dict, Any, Optional
import asyncio
from functools import partial
import evaluate
from collections import defaultdict
from dataclasses import dataclass

import hashlib

from tqdm import tqdm

from pydantic import BaseModel, Field, ValidationError
from json_repair import repair_json

from llama_cpp import Llama
from huggingface_hub import hf_hub_download

# Initialize metrics (run once)
try:
    bleu_metric = evaluate.load("bleu")
    rouge_metric = evaluate.load("rouge")
    meteor_metric = evaluate.load("meteor")
    bertscore_metric = evaluate.load("bertscore")
except:
    print("⚠️ Installing evaluation dependencies...")
    import subprocess
    subprocess.check_call(["pip", "install", "-q", "evaluate", "sacrebleu", "rouge-score", "bert-score", "nltk"])
    bleu_metric = evaluate.load("bleu")
    rouge_metric = evaluate.load("rouge")
    meteor_metric = evaluate.load("meteor")
    bertscore_metric = evaluate.load("bertscore")

Next, you'll need to download the model weights from HuggingFace.

Here's a list of models you can choose from: https://huggingface.co/models?pipeline_tag=text-generation&sort=trending&search=GGUF

Note: The model you select must be of type "GGUF"

GGUF is...
* binary file format for storing models for inference
* designed for fast loading and saving of models
* easy to use (with a few lines of code)
* mmap (memory mapping) compatibility: models can be loaded using mmap for fast loading and saving.



In [ ]:
# @title Select Large Language Model
selected_llm = 'Llama-3.2-3B-Instruct' # @param ["DeepSeek-R1-Distill-Qwen-1.5B-GGUF", "Qwen2.5-3B", "Llama-3.1-8B-Instruct", "Llama-3.2-3B-Instruct", "gpt-oss-20b"]

model_dic = {
    "DeepSeek-R1-Distill-Qwen-1.5B-GGUF":{"HF_REPO_NAME":"unsloth/DeepSeek-R1-Distill-Qwen-1.5B-GGUF","HF_MODEL_NAME":"DeepSeek-R1-Distill-Qwen-1.5B-Q8_0.gguf"},
    "Qwen2.5-3B":{"HF_REPO_NAME":"Qwen/Qwen2.5-3B-Instruct-GGUF","HF_MODEL_NAME":"qwen2.5-3b-instruct-q8_0.gguf"},
    "gpt-oss-20b":{"HF_REPO_NAME":"unsloth/gpt-oss-20b-GGUF", "HF_MODEL_NAME":"gpt-oss-20b-F16.gguf"},
    "Llama-3.1-8B-Instruct": {"HF_REPO_NAME": "unsloth/Llama-3.1-8B-Instruct-GGUF", "HF_MODEL_NAME": "Llama-3.1-8B-Instruct-BF16.gguf"},
    "Llama-3.2-3B-Instruct": {"HF_REPO_NAME": "bartowski/Llama-3.2-3B-Instruct-GGUF", "HF_MODEL_NAME": "Llama-3.2-3B-Instruct-f16.gguf"}
}

## Prompts

In [ ]:
class BaseAnswer(BaseModel):
    """Base class for all answer schemas."""
    answer: str

INSTRUCTIONS_ANSWER_JSON = """
CRITICAL: You MUST respond with ONLY valid JSON. Do not include any text before or after the JSON object.

Instructions:
- Your entire response must be a single JSON object starting with "{{" and ending with
- Do not include explanations, reasoning, or extra text
- The JSON must have exactly one key: "answer"
- The "answer" value must be at most 2 sentences (max 60 words)

Output format (JSON only):
{{
"answer": "<your concise factual answer>"
}}
END_JSON
"""

RAG_PROMPT = """
You are an expert assistant specialized in the EU AI Act, GDPR and regulatory compliance.
Your task is to answer the user's question as accurately and concisely as possible, strictly based on the resources provided below.

Question: {query}

Resources:
----------------------------------------
{context}
----------------------------------------

""" + INSTRUCTIONS_ANSWER_JSON

BASELINE_PROMPT = """
You are an expert assistant specialized in the EU AI Act, GDPR and regulatory compliance.
Your role is to provide accurate and concise answers.

CRITICAL: You MUST respond with ONLY valid JSON. Do not include any text before or after the JSON object.

Question: {query}

""" + INSTRUCTIONS_ANSWER_JSON

SELF_REFINEMENT_PROMPT = """
You are an expert assistant specialized in the EU AI Act, GDPR and regulatory compliance.
Your role is to Review and improve the given answers for accuracy. You will receive the starting question, the draft answer and the context text used by another expert.
Provide a more accurate and concise answer reviewing the input.

CRITICAL: You MUST respond with ONLY valid JSON. Do not include any text before or after the JSON object.

Question: {query}
Draft: {answer_text}

Resources:
----------------------------------------
{context}
----------------------------------------

""" + INSTRUCTIONS_ANSWER_JSON

REFINE_FROM_ISSUES_PROMPT = """
You are an expert assistant specialized in the EU AI Act, GDPR and regulatory compliance.
Revise the Draft to address the listed Issues using ONLY the Resources.

Rules:
- Only change what is needed to resolve Issues; otherwise keep the Draft wording stable.
- Do NOT add information not present in Resources.

Question: {query}

Draft:
{draft}

Issues (from auditor):
{issues_json}

Resources:
----------------------------------------
{context}
----------------------------------------
""" + INSTRUCTIONS_ANSWER_JSON

In [ ]:
INSTRUCTIONS_CRITIQUE_JSON = """
CRITICAL: Respond with ONLY valid JSON, no surrounding text.

Rules:
- Output must be exactly one JSON object.
- Keys MUST be exactly: "verdict", "issues"
- "verdict" is either "APPROVE" or "REVISE"
- "issues" is a list. If verdict is "APPROVE", issues MUST be [].
- Every issue MUST include an evidence quote copied from Resources; if you cannot quote, do not add the issue.

Return ONLY this JSON shape:
{{
  "verdict": "<APPROVE" | "REVISE>",
  "issues": [
    {{
      "type": "<unsupported" | "missing" | "contradiction>",
      "claim": "<what is wrong/missing>",
      "evidence_quote": "<verbatim quote from Resources that supports the critique>"
    }}
  ]
}}
END_JSON
"""

CRITIC_PROMPT = """
You are a strict compliance auditor.
Task: Evaluate whether the Draft answer is fully supported by the Resources and answers the Question.

Important:
- Do NOT use outside knowledge.
- If the Draft is correct and sufficiently supported, output verdict=APPROVE.
- If you request changes, each issue MUST include an evidence_quote copied from Resources.

Question: {query}

Draft:
{draft}

Resources:
----------------------------------------
{context}
----------------------------------------
""" + INSTRUCTIONS_CRITIQUE_JSON

## Utilities
Use this functions to optimize performance:
1. Cache llm responses for same queries
2. Retrieve topk_max one time for each question, then use only needed topk
3. Add checkpoints to restart inference

In [ ]:
# Cache base-generation per query (con e senza RAG)
base_generation_cache: Dict[str, str] = {}

def compute_query_hash(model_name:str, query: str, topk: int = 5, threshold: float = 0.0, use_graph: bool = False) -> str:
    query_str = f"{model_name}|{query}|{topk}|{threshold}|{use_graph}"
    return hashlib.md5(query_str.encode()).hexdigest()[:24]

async def get_base_generation(model_name:str, rag, query: str, llm_func, topk: int = 5, topk_max: int = 10, threshold: float = 0.0, use_graph: bool = False, temperature: float = 0.3) -> str:
    query_hash = compute_query_hash(model_name=model_name, query=query, topk=topk, threshold=threshold, use_graph=use_graph)

    if query_hash in base_generation_cache:
        return base_generation_cache[query_hash]

    retrieved_docs = await smart_retrieve(rag, query, topk_max=topk_max, threshold=threshold, use_graph=use_graph)
    context_docs = retrieved_docs[:topk]
    context_text = _join_context(context_docs)

    prompt = RAG_PROMPT.format(query=query, context=context_text)
    response = await askgenerator_raw(llm_func, prompt, temperature)
    answer = response.get("answer", "") if isinstance(response, dict) else str(response)

    base_generation_cache[query_hash] = answer

    return answer

In [ ]:
# -------- Retrieval helper: 1 FAISS call per query --------
# Cache base-generation per query (con e senza RAG)
base_retrieval_cache: Dict[str, str] = {}

def compute_rag_hash(query: str, topk_max: int = 10, threshold: float = 0.0, use_graph: bool = False) -> str:
    retrieval_str = f"{query}|{topk_max}|{threshold}|{str(use_graph)}"
    return hashlib.md5(retrieval_str.encode()).hexdigest()[:12]

async def smart_retrieve(rag, query: str, topk_max: int = 10, threshold: float = 0.0, use_graph: bool = False):
    """
    Retrieve topk_max una sola volta via FAISS+cosine, poi fai slice
    per usare topk=1,5,10 senza rifare retrieval.
    """
    query_hash = compute_rag_hash(query, topk_max, threshold, use_graph)

    if query_hash in base_retrieval_cache:
        return base_retrieval_cache[query_hash]

    return rag.retrieve_with_graph(query, topk=topk_max, threshold=threshold) if use_graph else rag.retrieve(query, topk=topk_max, threshold=threshold)

In [ ]:
def compute_config_hash(model_name: str, pattern_name: str, topk: int, limit: int) -> str:
    config_str = f"{model_name}|{pattern_name}|{topk}|{limit}"
    return hashlib.md5(config_str.encode()).hexdigest()[:8]

def load_checkpoint(output_dir: str) -> Dict[str, bool]:
    checkpoint_file = os.path.join(output_dir, "checkpoint.json")
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, 'r') as f:
            return json.load(f)
    return {}

def save_checkpoint(output_dir: str, checkpoint: Dict[str, bool]):
    checkpoint_file = os.path.join(output_dir, "checkpoint.json")
    with open(checkpoint_file, 'w') as f:
        json.dump(checkpoint, f, indent=2)

def _join_context(docs: List[Dict[str, Any]]) -> str:
    return "\n".join(d["text"] for d in docs if d.get("text"))

In [ ]:
def extract_json_from_text(
    text: str,
    expected_keys: Optional[list] = None,
    allow_arrays: bool = False
) -> Dict[str, Any]:
    """
    Extracts and parses JSON from a text string with advanced fallback logic.

    This function handles multiple scenarios:
    - JSON embedded in text with surrounding content
    - Thinking tags that need to be stripped
    - Malformed JSON that can be repaired
    - JSON objects and optionally JSON arrays
    - Plain text when no JSON is found
    - Schema validation when expected keys are provided

    Args:
        text: The input text that may contain JSON. Can have prefixes, suffixes,
              or thinking tags like </think>.
        expected_keys: Optional list of keys that should be present in the parsed
                       JSON object. If provided, validates the parsed JSON contains
                       these keys. Example: ["answer", "confidence"]
        allow_arrays: If True, accepts JSON arrays as valid responses. If False,
                      only JSON objects are considered valid JSON.

    Returns:
        Dict[str, Any]: A dictionary with "answer" key containing:
            - The value of parsed["answer"] if JSON object with "answer" key
            - The entire parsed object/array if valid JSON without "answer" key
            - The raw JSON string if parsing fails after repair attempts
            - The plain text if no JSON structure is found

    Example:
        >>> text = 'Here is the result: {"answer": "42", "confidence": 0.95}'
        >>> result = extract_json_from_text(text, expected_keys=["answer"])
        >>> print(result)
        {'answer': '42'}

        >>> text = '[1, 2, 3]'
        >>> result = extract_json_from_text(text, allow_arrays=True)
        >>> print(result)
        {'answer': [1, 2, 3]}
    """
    print(f"    🟡 extract_json_from_text: CALLED with text='{text[:100]}...'")

    # Remove thinking tags if present
    if "</think>" in text:
        text = text.split("</think>", 1)[-1].strip()

    # Try to find JSON object
    start_obj = text.find("{")
    end_obj = text.rfind("}")

    # Try to find JSON array if allowed
    start_arr = text.find("[") if allow_arrays else -1
    end_arr = text.rfind("]") if allow_arrays else -1

    # Determine which JSON structure to extract (prefer object over array)
    json_str = None
    if start_obj != -1 and end_obj != -1 and end_obj > start_obj:
        json_str = text[start_obj:end_obj+1]
        print(f"    🟡 extract_json_from_text: found JSON object='{json_str[:100]}...'")
    elif start_arr != -1 and end_arr != -1 and end_arr > start_arr:
        json_str = text[start_arr:end_arr+1]
        print(f"    🟡 extract_json_from_text: found JSON array='{json_str[:100]}...'")

    if json_str:
        # Try to parse JSON
        parsed = _parse_json_with_repair(json_str)

        if parsed is not None:
            # Validate expected keys if provided
            if expected_keys and isinstance(parsed, dict):
                missing_keys = [key for key in expected_keys if key not in parsed]
                if missing_keys:
                    print(f"    ⚠️  extract_json_from_text: Missing expected keys: {missing_keys}")

            # Normalize response
            if isinstance(parsed, dict) and "answer" in parsed:
                return {"answer": parsed["answer"]}
            elif isinstance(parsed, (dict, list)):
                return {"answer": parsed}
            else:
                return {"answer": json_str}
        else:
            # Parsing failed even with repair
            print(f"    🔴 extract_json_from_text: All parsing attempts failed")
            return {"answer": text.strip()}

    # No JSON found: return plain text as answer
    print(f"    🟡 extract_json_from_text: No JSON found, using plain text")
    return {"answer": text.strip()}


def _parse_json_with_repair(json_str: str) -> Optional[Union[Dict, list]]:
    """
    Helper function to parse JSON with repair fallback.

    Attempts to parse JSON string using standard json.loads first. If that fails
    with JSONDecodeError, attempts to repair the JSON using json_repair library.

    Args:
        json_str: The JSON string to parse.

    Returns:
        Optional[Union[Dict, list]]: The parsed JSON object/array if successful,
                                     None if all parsing attempts fail.
    """
    # Try standard JSON parsing
    try:
        parsed = json.loads(json_str)
        print(f"    ✅ _parse_json_with_repair: Successfully parsed JSON")
        return parsed
    except json.JSONDecodeError as e:
        print(f"    🟡 _parse_json_with_repair: Standard JSON parse failed: {e}")

    # Try repair_json as fallback
    try:
        from json_repair import repair_json
        repaired_str = repair_json(json_str)
        parsed = json.loads(repaired_str)
        print(f"    ✅ _parse_json_with_repair: Successfully repaired and parsed JSON")
        return parsed
    except Exception as e:
        print(f"    🔴 _parse_json_with_repair: repair_json failed: {e}")
        return None

## Patterns
To implement the agentic patterns while integrating your RAG system, we will modify the `ask_generator` logic to support **Routing**, **Collaboration (Debate)**, and **Self-Refinement**.

The following implementation assumes the existence of a `rag.retrieve(query, topk, threshold)` function and adapts the code from your `QA.py` and the provided markdown patterns.

### Implementation Strategy
The core of these patterns involves shifting from a single LLM call to a multi-step workflow. We will wrap these in a new `AgentEvaluator` or extend your existing generation loop.

### 0. Baseline (with and without Rag)

In [ ]:
async def baseline_pattern(query: str, llm_func) -> str:
    """Pattern 0: Baseline LLM without RAG (topk ignored)"""
    prompt = BASELINE_PROMPT.format(query=query)
    response = await askgenerator_raw(llm_func, prompt)
    return response.get("answer", "") if isinstance(response, dict) else str(response)


async def rag_pattern(model_name: str, query: str, rag, llm_func, topk: int = 5, topk_max: int = 20, threshold: float = 0.0) -> str:
    """Pattern 1: LLM + RAG"""
    return await get_base_generation(model_name=model_name, rag=rag, query=query, llm_func=llm_func, topk=topk, topk_max=topk_max, threshold=threshold, use_graph=False)


async def rag_with_graph_pattern(model_name: str, query: str, rag, llm_func, topk: int = 5, topk_max: int = 20, threshold: float = 0.0) -> str:
    """Pattern 1.5: LLM + RAG + GRAPH"""
    return await get_base_generation(model_name=model_name, rag=rag, query=query, llm_func=llm_func, topk=topk, topk_max=topk_max, threshold=threshold, use_graph=True)

### 1. Routing Pattern (Specialization)
This pattern uses a "Router" to select the most appropriate expert model or prompt based on the query domain.

In [ ]:
async def routing_rag_pattern(model_name: str, query: str, rags_dict: Dict[str, Any], llm_func, topk: int = 5, topk_max: int = 20, threshold: float = 0.0, use_graph: bool = False) -> str:
    """Pattern 2: LLM + RAG + Routing (domain classification)"""
    # Route query to domain
    route_prompt = f"Classify this query as 'GDPR', 'AIACT', or 'GENERAL': {query}. Critical: Output only the domain as a single word string!"
    domain_raw = await llm_func(route_prompt, max_tokens=10, stop=[".", "\n"])
    domain = (domain_raw.get("choices", [{}])[0].get("text", "GENERAL")).strip().upper()

    domanin = "GDPR" if "GDPR" in domain else "AIACT" if "AIACT" in domain else "GENERAL"

    # Domain-specific retrieval threshold
    threshold_map = {"GDPR": 0.4, "AIACT": 0.5, "GENERAL": 0.3}
    threshold = threshold_map.get(domain, 0.3)
    rag = rags_dict.get(domain, None)

    if rag is None:
        return "Empty answer. Couldn't detect domain"

    return await get_base_generation(model_name=model_name, rag=rag, query=query, llm_func=llm_func, topk=topk, topk_max=topk_max, threshold=threshold, use_graph=use_graph)


### 2. Collaboration Pattern (Multi-Agent Debate)
This pattern involves a "Generator" creating an initial response and a "Critic" reviewing it for errors or missing citations.

In [ ]:
async def collaboration_rag_pattern(
    model_name: str,
    query: str,
    rag,
    llm_func,
    topk: int = 5,
    topk_max: int = 20,
    threshold: float = 0.0,
    use_graph: bool = False,
    max_rounds: int = 2,
) -> str:
    # Retrieve ONCE and reuse everywhere (prevents drift/mismatch).
    retrieved_docs = await smart_retrieve(
        rag,
        query,
        topk_max=topk_max,
        threshold=threshold,
        use_graph=use_graph,
    )
    context_docs = retrieved_docs[:topk]
    context_text = _join_context(context_docs)

    # Initial answer (deterministic)
    answer_text = await get_base_generation(
        model_name=model_name,
        rag=rag,
        query=query,
        llm_func=llm_func,
        topk=topk,
        topk_max=topk_max,
        threshold=threshold,
        use_graph=use_graph
    )

    for _ in range(max_rounds):
        critic_prompt = CRITIC_PROMPT.format(
            query=query,
            draft=answer_text,
            context=context_text,
        )

        critique = await askgenerator_raw(
            llm_func=llm_func,
            prompt=critic_prompt,
            max_tokens=300,
            temperature=0.0,   # important for stability,
            expected_keys=["verdict","issues"],
            allow_arrays=True
        )

        payload = critique.get("answer")
        if not isinstance(payload, dict):
            payload = {}  # fallback

        verdict = payload.get("verdict", "REVISE")
        issues = payload.get("issues", payload)

        if verdict == "APPROVE":
            break

        refine_prompt = REFINE_FROM_ISSUES_PROMPT.format(
            query=query,
            draft=answer_text,
            issues_json=json.dumps(issues, ensure_ascii=False),
            context=context_text,
        )
        refined = await askgenerator_raw(llm_func, refine_prompt)
        answer_text = refined.get("answer", "") if isinstance(refined, dict) else str(refined)

    return answer_text

### 3. Self-Refinement Pattern (Iterative Reasoning)
The model plans its steps, retrieves evidence, and critiques its own draft iteratively to reduce hallucination.

In [ ]:
async def self_refinement_rag_pattern(model_name: str, query: str, rag, llm_func, topk: int = 5, max_iters: int = 2, topk_max: int = 20, threshold: float = 0.0, use_graph: bool = True) -> str:
    """Pattern 4: LLM + RAG + Self-Refinement (iterative improvement)"""
    retrieved_docs = await smart_retrieve(rag, query, topk_max=topk_max, threshold=0.0, use_graph=True)
    context_docs = retrieved_docs[:topk]

    context_text = "\n".join([d["text"] for d in context_docs])

    # Initial draft
    answer_text = await get_base_generation(model_name, rag, query, llm_func, topk=topk, topk_max=topk_max, threshold=threshold, use_graph=use_graph)

    # Self-refinement loop
    for _ in range(max_iters):
        refine_prompt = SELF_REFINEMENT_PROMPT.format(query=query, answer_text=answer_text, context=context_text)
        refined = await askgenerator_raw(llm_func, refine_prompt, temperature=0.3)
        new_answer = refined.get("answer", "") if isinstance(refined, dict) else str(refined)

        if new_answer.strip() == answer_text.strip():
            break
        answer_text = new_answer

    return answer_text

## Inference

In [ ]:
async def call_llm_with_retry(
    llm_func,
    prompt: str,
    temperature: float = 0.5,
    max_tokens: int = 512,
    retries: int = 2
) -> str:
    """
    Calls the LLM function with retry logic and extracts raw text response.

    This function handles various LLM response formats (OpenAI-compatible dict
    or plain string) and automatically retries on failure.

    Args:
        llm_func: Async callable that accepts prompt, temperature, max_tokens,
                  and stop parameters. Should return either a string or an
                  OpenAI-compatible response dict with 'choices' field.
        prompt: The text prompt to send to the LLM.
        temperature: Controls randomness in generation. Lower values (e.g., 0.2)
                     make output more deterministic, higher values (e.g., 0.8)
                     make it more creative. Range: 0.0-1.0.
        max_tokens: Maximum number of tokens to generate in the response.
        retries: Number of retry attempts if the call fails. Total attempts
                 will be retries + 1.

    Returns:
        str: The extracted text response from the LLM.

    Raises:
        Exception: If all retry attempts fail, the last exception is re-raised.

    Example:
        >>> async def my_llm(prompt, temperature, max_tokens, stop):
        ...     return {"choices": [{"text": "Hello world"}]}
        >>> result = await call_llm_with_retry(my_llm, "Say hello")
        >>> print(result)
        'Hello world'
    """
    print(f"    🟡 call_llm_with_retry: CALLED")

    for attempt in range(retries + 1):
        try:
            print(f"    🟡 call_llm_with_retry: attempt {attempt+1}, calling llm_func...")
            text_response = await llm_func(
                prompt=prompt,
                temperature=temperature,
                max_tokens=max_tokens,
                stop=["END_JSON"]
            )

            print(f"    🟡 call_llm_with_retry: got response={text_response} type={type(text_response)}")

            # Extract text from OpenAI-compatible response
            if isinstance(text_response, dict) and "choices" in text_response:
                text = text_response["choices"][0].get("text", "")
            elif isinstance(text_response, str):
                text = text_response
            else:
                text = str(text_response)

            print(f"    🟡 call_llm_with_retry: extracted text='{text[:100]}...'")
            return text

        except Exception as e:
            print(f"    🔴 call_llm_with_retry: attempt {attempt+1} failed: {e}")
            if attempt == retries:
                raise
            continue

    raise Exception("Failed to generate response after all retries")


async def askgenerator_raw(
    llm_func,
    prompt: str,
    temperature: float = 0.5,
    max_tokens: int = 512,
    retries: int = 2,
    expected_keys: Optional[list] = ["answer"],
    allow_arrays: bool = False
) -> Dict[str, Any]:
    """
    Orchestrates LLM call and JSON extraction with comprehensive error handling.

    This is the main entry point that combines LLM invocation with intelligent
    response parsing. It handles the complete flow: call LLM → extract text →
    parse JSON → validate → normalize response.

    Args:
        llm_func: Async callable that accepts prompt, temperature, max_tokens,
                  and stop parameters. Should return either a string or an
                  OpenAI-compatible response dict.
        prompt: The text prompt to send to the LLM.
        temperature: Controls randomness (0.0-1.0). Lower is more deterministic.
        max_tokens: Maximum tokens to generate.
        retries: Number of retry attempts on failure.
        expected_keys: Optional list of keys to validate in the JSON response.
                       Example: ["answer", "reasoning"]
        allow_arrays: If True, accept JSON arrays as valid responses.

    Returns:
        Dict[str, Any]: A dictionary with "answer" key containing the extracted
                        result. In case of errors, returns {"answer": "Error: ..."}

    Example:
        >>> async def my_llm(prompt, **kwargs):
        ...     return '{"answer": "The capital is Paris", "confidence": 0.99}'
        >>> result = await askgenerator_raw(
        ...     my_llm,
        ...     "What is the capital of France?",
        ...     expected_keys=["answer"]
        ... )
        >>> print(result)
        {'answer': 'The capital is Paris'}
    """
    print(f"    🟡 askgenerator_raw: CALLED")

    try:
        text = await call_llm_with_retry(
            llm_func=llm_func,
            prompt=prompt,
            temperature=temperature,
            max_tokens=max_tokens,
            retries=retries
        )
        return extract_json_from_text(
            text=text,
            expected_keys=expected_keys,
            allow_arrays=allow_arrays
        )

    except Exception as e:
        print(f"    🔴 askgenerator_raw: failed with error: {e}")
        return {"answer": f"Error: {str(e)}"}

## Metrics computation

In [ ]:
# ============================================================================
# Metrics Computation
# ============================================================================

def normalize_text(text: str) -> str:
    """Basic text normalization"""
    return text.lower().strip()

def compute_em(pred: str, ref: str) -> float:
    return 1.0 if normalize_text(pred) == normalize_text(ref) else 0.0

def compute_f1(pred: str, ref: str) -> float:
    pt = normalize_text(pred).split()
    rt = normalize_text(ref).split()
    if not pt or not rt:
        return 0.0
    common = set(pt) & set(rt)
    if not common:
        return 0.0
    precision = len(common) / len(pt)
    recall = len(common) / len(rt)
    return 2 * precision * recall / (precision + recall)


def compute_all_metrics(predictions: List[str], references: List[str]) -> Dict[str, float]:
    """Compute all evaluation metrics"""
    results = defaultdict(float)
    n = len(predictions)

    if n == 0:
        return results

    # EM and F1 (per-sample, then averaged)
    try:
      em_scores = [compute_em(p, r) for p, r in zip(predictions, references)]
      f1_scores = [compute_f1(p, r) for p, r in zip(predictions, references)]
      results["em"] = sum(em_scores) / n
      results["f1"] = sum(f1_scores) / n
    except Exception as e:
        print(f"⚠️ EM/F1 computation failed: {e}")
        results["em"] = results["f1"] = 0.0

    # BLEU
    try:
        bleu_result = bleu_metric.compute(
            predictions=predictions,
            references=[[r] for r in references]  # BLEU needs list of lists
        )
        results["bleu"] = bleu_result["bleu"]
    except Exception as e:
        print(f"⚠️ BLEU computation failed: {e}")
        results["bleu"] = 0.0

    # ROUGE
    try:
        rouge_result = rouge_metric.compute(
            predictions=predictions,
            references=references
        )
        results["rouge1"] = rouge_result["rouge1"]
        results["rouge2"] = rouge_result["rouge2"]
        results["rougeL"] = rouge_result["rougeL"]
    except Exception as e:
        print(f"⚠️ ROUGE computation failed: {e}")
        results["rouge1"] = results["rouge2"] = results["rougeL"] = 0.0

    # METEOR - needs special format
    try:
        # METEOR expects predictions and references as simple lists
        meteor_result = meteor_metric.compute(
            predictions=predictions,
            references=references
        )
        results["meteor"] = meteor_result["meteor"]
    except Exception as e:
        # Fallback: try with wrapped format
        try:
            meteor_formatted_preds = [{"id": str(i), "prediction_text": p} for i, p in enumerate(predictions)]
            meteor_formatted_refs = [{"id": str(i), "answers": {"text": [r], "answer_start": [0]}} for i, r in enumerate(references)]
            meteor_result = meteor_metric.compute(
                predictions=meteor_formatted_preds,
                references=meteor_formatted_refs
            )
            results["meteor"] = meteor_result["meteor"]
        except Exception as e2:
            print(f"⚠️ METEOR computation failed (both formats): {e2}")
            results["meteor"] = 0.0

    # BERTScore - reduce batch size for memory efficiency
    try:
        bertscore_result = bertscore_metric.compute(
            predictions=predictions,
            references=references,
            lang="en",
            model_type="distilbert-base-uncased",  # Lighter model
            batch_size=8  # Smaller batch to avoid memory issues
        )
        results["bertscore_f1"] = sum(bertscore_result["f1"]) / len(bertscore_result["f1"])
        results["bertscore_p"] = sum(bertscore_result["precision"]) / len(bertscore_result["precision"])
    except Exception as e:
        print(f"⚠️ BERTScore computation failed: {e}")
        results["bertscore_f1"] = 0.0
        results["bertscore_p"] = 0.0

    return dict(results)

## Evaluation Pipeline

In [ ]:
# ============================================================================
# Main Evaluation Pipeline
# ============================================================================
async def evaluate_model_pattern(
    model_name: str,
    pattern_name: str,
    pattern_func: Callable,
    llm_func: Callable,
    rags_dict: Optional[Any],
    domain: str,
    qa_pairs: List[Dict],
    topk: int,
    topk_max: int,
    limit: Optional[int] = None
) -> Dict[str, Any]:
    """Evaluate a single model×pattern×topk combination"""

    print(f"🟢 START evaluate_model_pattern: {model_name} | {pattern_name} | topk={topk}")

    predictions = []
    references = []

    rag = rags_dict[domain] # domain specific rag

    qa_subset = qa_pairs[:limit] if limit else qa_pairs
    print(f"🟢 Processing {len(qa_subset)} samples...")

    for i, qa in enumerate(tqdm(qa_subset, desc=f"{model_name}|{pattern_name}|k={topk}")):
        query = qa["question"]

        reference = str(qa.get("answer", "")).strip()
        if not reference:
            reference = "No reference available"

        print(f"🔵 Sample {i}: calling pattern...")

        try:
            # Call pattern-specific function with topk
            if pattern_name == "baseline":
                pred = await asyncio.wait_for(
                    pattern_func(query, llm_func),
                    timeout=60.0
                )
            elif pattern_name == "rag_routing":
                pred = await asyncio.wait_for(
                    pattern_func(model_name, query, rags_dict, llm_func, topk=topk, topk_max=topk_max),
                    timeout=60.0
                )
            else:
                pred = await asyncio.wait_for(
                    pattern_func(model_name, query, rag, llm_func, topk=topk, topk_max=topk_max),
                    timeout=60.0
                )
            predictions.append(pred)
            references.append(reference)

            print(f"🟢 Sample {i}: got prediction type={type(pred)}, value={pred[:50] if isinstance(pred, str) else pred}")

        except asyncio.TimeoutError:
            print(f"  ⏱️ Timeout on sample {i}")
            continue
        except Exception as e:
            print(f"  ⚠️ Error on sample {i}: {e}")
            continue

    # Creating predictions dataframe
    df_predictions = pd.DataFrame({"question": [qa["question"] for qa in qa_subset], "prediction": predictions, "reference": references,
                                  "model": [model_name]*len(predictions), "pattern": [pattern_name]*len(predictions), "topk": [topk]*len(predictions),})


    # Compute metrics
    print(f"🟢 Computing metrics on {len(predictions)} predictions...")
    for i in range(len(predictions)):
        if isinstance(predictions[i], dict):
            predictions[i] = predictions[i].get("answer", "")
        if not isinstance(predictions[i], str):
            predictions[i] = str(predictions[i])

    print(f"🟢 Computing metrics on {len(predictions)} predictions...")
    metrics = compute_all_metrics(predictions, references)

    result = {
        "model": model_name,
        "pattern": pattern_name,
        "topk": topk,
        "n_samples": len(predictions),
        **metrics
    }

    print(f"✅ {model_name} | {pattern_name} | topk={topk} | F1={metrics['f1']:.3f} | ROUGE-L={metrics['rougeL']:.3f}")

    return result, df_predictions


async def run_full_evaluation(
    models: List[str],
    rags_dict: Any,
    domain: str,
    qa_pairs: List[Dict],
    topk_values: List[int] = [1, 5, 10],
    output_dir: str = "./evaluation_results",
    limit_per_model: Optional[int] = 50,
    resume: bool = False
):
    """
    Run complete evaluation grid: models × patterns × topk

    Args:
        models: List of model names to test
        rags_dict: Dictionary with rags per domain
        domain: String representing the domain that is being tested
        qa_pairs: Q&A dataset
        topk_values: List of topk values to test (default: [1, 5, 10])
        output_dir: Where to save results
        limit_per_model: Max samples per model (for quick testing)
    """
    timestamp = get_timestamp()

    checkpoint_dir = output_dir
    checkpoint = load_checkpoint(checkpoint_dir) if resume else {}
    executions = checkpoint.get("executions", [])
    if not executions:
      checkpoint["executions"] = [timestamp]
    else:
      checkpoint["executions"].append(timestamp)
    print(f"Checkpoint: {checkpoint}")

    output_dir = os.path.join(output_dir, timestamp)
    os.makedirs(output_dir, exist_ok=True)

    # Validate models exist in model_dic
    invalid_models = [m for m in models if m not in model_dic]
    if invalid_models:
        raise ValueError(f"Unknown models: {invalid_models}. Available: {list(model_dic.keys())}")

    # Validate topk values
    if not all(k > 0 for k in topk_values):
        raise ValueError(f"All topk values must be > 0, got: {topk_values}")

    # Pattern registry
    patterns = {
        "baseline": baseline_pattern,
        "rag": rag_pattern,
        "rag_graphs": rag_with_graph_pattern,
        "rag_routing": routing_rag_pattern,
        "rag_collaboration": collaboration_rag_pattern,
        "rag_self_refinement": self_refinement_rag_pattern,
    }

    all_results = []

    for model_name in models:
        print(f"\n{'='*60}")
        print(f"🚀 Testing Model: {model_name}")
        print(f"{'='*60}\n")

        # Initialize model
        try:
            model_config = model_dic[model_name]
            HF_REPO_NAME = model_config["HF_REPO_NAME"]
            HF_MODEL_NAME = model_config["HF_MODEL_NAME"]

            model_path = hf_hub_download(
                repo_id=HF_REPO_NAME,
                filename=HF_MODEL_NAME,
                local_dir="./models"
            )

            llm = Llama(
                model_path=model_path,
                n_threads=2,
                n_batch=512,
                n_gpu_layers=40,
                n_ctx=4096,
            )

            # Create async wrapper
            async def llm_func(prompt, **kwargs):
              """Async wrapper for llama.cpp model"""
              loop = asyncio.get_running_loop()

              # Estrai parametri comuni per evitare conflitti
              params = {
                  "prompt": prompt,
                  "stop": kwargs.pop("stop", []),
                  "max_tokens": kwargs.pop("max_tokens", 512),
                  "temperature": kwargs.pop("temperature", 0.3),
                  **kwargs  # Altri parametri rimanenti
              }

              call_func = partial(llm, **params)
              response = await loop.run_in_executor(None, call_func)
              return response

        except Exception as e:
            print(f"❌ Failed to load {model_name}: {e}")
            continue

        # Test all patterns × topk combinations
        for pattern_name, pattern_func in patterns.items():
            # Baseline doesn't use RAG, so test only once
            if pattern_name == "baseline":
                topk_list = [topk_values[0]]  # Use first topk value as placeholder
            else:
                topk_list = topk_values

            topk_max = max(topk_list)

            for topk in topk_list:
                config_id = compute_config_hash(model_name, pattern_name, topk, limit_per_model)

                if config_id in checkpoint:
                    print(f"⏭️ Skipping {model_name} | {pattern_name} | topk={topk}")
                    continue

                try:
                    result, df_predictions = await evaluate_model_pattern(
                        model_name=model_name,
                        pattern_name=pattern_name,
                        pattern_func=pattern_func,
                        llm_func=llm_func,
                        rags_dict=rags_dict,
                        domain=domain,
                        qa_pairs=qa_pairs,
                        topk=topk,
                        topk_max=topk_max,
                        limit=limit_per_model
                    )
                    all_results.append(result)

                    # Save incremental results
                    df_results = pd.DataFrame(all_results)
                    df_results.to_csv(f"{output_dir}/results_incremental.csv", index=False)

                    df_predictions.to_csv(f"{output_dir}/predictions_incremental.csv", index=False)

                    checkpoint[config_id] = True
                    save_checkpoint(checkpoint_dir, checkpoint)
                except Exception as e:
                    print(f"❌ Failed {model_name} | {pattern_name} | topk={topk}: {e}")
                    continue

        # Free GPU memory
        del llm
        import gc
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Final results
    df_final = pd.DataFrame(all_results)
    if not df_final.empty:
      # df_final.to_csv(f"{output_dir}/results_final.csv", index=False)

      # Summary by pattern (average across models)
      print(f"\n{'='*60}")
      print(f"📊 Best Configurations")
      print(f"{'='*60}\n")
      best_by_pattern = df_final.loc[df_final.groupby('pattern')['f1'].idxmax()]
      print(best_by_pattern[['pattern', 'model', 'topk', 'f1', 'rougeL']].to_string(index=False))

      # Summary by topk (for RAG patterns only)
      print(f"\n{'='*60}")
      print(f"📊 TopK Impact (RAG patterns only)")
      print(f"{'='*60}\n")
      rag_only = df_final[df_final['pattern'].str.contains('rag')]
      topk_impact = rag_only.groupby('topk')[['f1', 'rougeL', 'bertscore_f1']].mean()
      print(topk_impact.round(3))

      # Summary by topk
      print(f"\n{'='*60}")
      print(f"📊 Summary by TopK")
      print(f"{'='*60}\n")
      summary = df_final.groupby(['pattern', 'topk'])[['f1', 'rougeL', 'bertscore_f1']].mean()
      print(summary.round(3))

      print(f"\n{'='*60}")
      print(f"✅ Evaluation Complete! Results saved to {output_dir}")
      print(f"{'='*60}\n")

    return df_final

## Run test

In [ ]:
# Define models to test
models_to_test = [
    "DeepSeek-R1-Distill-Qwen-1.5B-GGUF",
    "Qwen2.5-3B",
    "Llama-3.2-3B-Instruct",
    "gpt-oss-20b",            # uncomment if you have enough VRAM
]

In [ ]:
# Initialize RAG (use your existing initialization)
rags_dict = {rags[rag_type]["config"].DOMAIN:rags[rag_type]["rag"] for rag_type in rags}
print(rags_dict)

for rag_type in rags:
  config = rags[rag_type]["config"]

  # Load dataset
  qa_pairs = qa_pairs_dict[config.DOMAIN]

  # Run evaluation with custom topk values
  results_df = await run_full_evaluation(
      models=models_to_test,
      rags_dict=rags_dict,
      domain=config.DOMAIN,
      qa_pairs=qa_pairs,
      topk_values=[1, 5, 10],  # Test with topk = 1, 5, 10
      output_dir=config.GENERATOR_RESULTS_DIR,
      limit_per_model=None,
      resume = True
  )

  results_df.head()

## Explore Results

In [ ]:
import pandas as pd
import os

def load_final_csv(checkpoint_dir: str = "path/to/checkpoint_file"):
    checkpoint_json = load_checkpoint(checkpoint_dir)
    executions_paths = [os.path.join(checkpoint_dir, execution, "results_incremental.csv") for execution in checkpoint_json["executions"]]

    df_final = pd.DataFrame()
    for exec in executions_paths:
      df_exec = pd.read_csv(exec)
      df_final = pd.concat([df_final, df_exec])

    return df_final, executions_paths

def load_and_explore_results(checkpoint_dir: str = "path/to/checkpoint_file") -> pd.DataFrame:
    # ---------------------------------------------------------------------
    # 1. Load CSV
    # ---------------------------------------------------------------------
    df_final, csv_paths = load_final_csv(checkpoint_dir)

    # ---------------------------------------------------------------------
    # 2. Basic inspection (keeping existing)
    # ---------------------------------------------------------------------
    print("\n" + "=" * 60)
    print("📂 Loaded DataFrame")
    print("=" * 60 + "\n")
    print(f"Paths: {csv_paths}")
    print(f"Shape: {df_final.shape}")
    print("\nColumns:", list(df_final.columns))
    print("\nHead:\n", df_final.head())

    # ---------------------------------------------------------------------
    # 3. Value counts for key categorical columns
    # ---------------------------------------------------------------------
    if "pattern" in df_final.columns:
        print("\n" + "=" * 60)
        print("📊 Pattern distribution")
        print("=" * 60 + "\n")
        print(df_final["pattern"].value_counts())

    if "model" in df_final.columns:
        print("\n" + "=" * 60)
        print("📊 Model distribution")
        print("=" * 60 + "\n")
        print(df_final["model"].value_counts())

    if "topk" in df_final.columns:
        print("\n" + "=" * 60)
        print("📊 TopK distribution")
        print("=" * 60 + "\n")
        print(df_final["topk"].value_counts().sort_index())

    # ---------------------------------------------------------------------
    # 4. PIVOT TABLE like the image (Model x Method, metrics as subcolumns)
    # ---------------------------------------------------------------------
    print("\n" + "=" * 120)
    print("📊 PIVOT TABLE: Model x Method (All Metrics)")
    print("=" * 120 + "\n")

    # Prepare data for pivot (Method column might be 'pattern')
    method_col = 'pattern' if 'pattern' in df_final.columns else None

    if method_col and 'model' in df_final.columns:
        # Pivot: Models as rows, Methods as columns, mean metrics
        pivot_data = df_final.pivot_table(
            index='model',
            columns=method_col,
            values=['em', 'f1', 'bleu', 'rouge1', 'rouge2', 'rougeL', 'meteor', 'bertscore_f1'],
            aggfunc='mean',
            fill_value=0
        ).round(3)

        print(pivot_data.to_string())
    else:
        print("Cannot create pivot: missing 'model' or 'pattern' columns.")

    # ---------------------------------------------------------------------
    # 5. Best configuration per pattern (existing)
    # ---------------------------------------------------------------------
    print("\n" + "=" * 60)
    print("🏆 Best Configurations per Pattern (by F1)")
    print("=" * 60 + "\n")

    if 'pattern' in df_final.columns and 'f1' in df_final.columns:
        best_by_pattern = df_final.loc[df_final.groupby("pattern")["f1"].idxmax()]
        cols_best = [
            c for c in [
                "pattern", "model", "topk",
                "em", "f1", "bleu", "rouge1", "rouge2", "rougeL",
                "meteor", "bertscore_f1"
            ] if c in best_by_pattern.columns
        ]
        print(best_by_pattern[cols_best].to_string(index=False))

    # ---------------------------------------------------------------------
    # 6. Summary by (pattern, topk) with ALL metrics (existing)
    # ---------------------------------------------------------------------
    print("\n" + "=" * 60)
    print("📊 Summary by Pattern & TopK (All Metrics)")
    print("=" * 60 + "\n")

    group_keys = []
    if "pattern" in df_final.columns:
        group_keys.append("pattern")
    if "topk" in df_final.columns:
        group_keys.append("topk")

    all_metric_cols = [
        c for c in [
            "em", "f1", "bleu", "rouge1", "rouge2", "rougeL",
            "meteor", "bertscore_f1"
        ] if c in df_final.columns
    ]

    if group_keys and all_metric_cols:
        summary_all = df_final.groupby(group_keys)[all_metric_cols].mean()
        print(summary_all.round(3))

    # ---------------------------------------------------------------------
    # 7. Correlations between metrics (optional exploration)
    # ---------------------------------------------------------------------
    if all_metric_cols:
        print("\n" + "=" * 60)
        print("📈 Metric Correlations")
        print("=" * 60 + "\n")
        corr = df_final[all_metric_cols].corr()
        print(corr.round(3))

    # ---------------------------------------------------------------------
    # 8. Return the loaded DataFrame
    # ---------------------------------------------------------------------
    print("\n" + "=" * 60)
    print(f"✅ Exploration Complete! Loaded from {csv_paths}")

    final_csv_path = os.path.join(checkpoint_dir, "results_final.csv")
    df_final.to_csv(final_csv_path, index=False)
    print(f"✅ Final dataframe saved in: {final_csv_path}")
    print("=" * 60 + "\n")

    return df_final


In [ ]:
# Usage
df = load_and_explore_results(gdpr_config.GENERATOR_RESULTS_DIR)

In [ ]:
# Usage
df = load_and_explore_results(aiact_config.GENERATOR_RESULTS_DIR)